# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @id
record_sets = dataset.metadata.record_sets
print("Record sets in the dataset:")
for rs in record_sets:
    print(f"- Record set @id: {rs['@id']}, Name: {rs.get('name', 'No name')}")

# List fields and columns for each record set
for rs in record_sets:
    print(f"\nFields for Record Set @id {rs['@id']}: {rs.get('name', 'No name')}")
    fields = rs.get('fields', [])
    for field in fields:
        print(f"    - Field @id: {field['@id']} | Name: {field.get('name', '')}")
        columns = field.get('columns', [])
        for col in columns:
            print(f"        - Column @id: {col['@id']} | Name: {col.get('name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}
# Load records from each record set
for record_set_id in record_set_ids:
    # Use the mlcroissant .records() API referencing by @id
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show columns and first few rows from one of the record sets
# Choose the first record set as default
first_record_set_id = record_set_ids[0]
print(f"Columns for record set {first_record_set_id}:\n", dataframes[first_record_set_id].columns.tolist())
dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For EDA, select a numeric field (e.g. GAD-7 score) and a group field (e.g. gender)
# Replace below with actual field @id as found above
# Example field and group field IDs (you may need to change accordingly):
numeric_field_id = None
group_field_id = None

# Find candidate numeric and group fields from the first record set
df = dataframes[first_record_set_id]
for col in df.columns:
    # Heuristically select GAD-7 or PHQ-9 as numeric field
    if 'GAD' in col or 'PHQ' in col or 'score' in col:
        numeric_field_id = col
    # Heuristically select gender as group field
    if 'gender' in col or 'Gender' in col:
        group_field_id = col
if numeric_field_id is None:
    numeric_field_id = df.select_dtypes(include='number').columns[0] if len(df.select_dtypes(include='number').columns) > 0 else df.columns[0]
if group_field_id is None:
    group_field_id = 'gender' if 'gender' in df.columns else df.columns[0]

print(f"Selected numeric field @id: {numeric_field_id}")
print(f"Selected group field @id: {group_field_id}")

# Filter: keep records with numeric_field > threshold
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group field and get mean
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Simple visualization example: Histogram of selected numeric field and boxplot by group field
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

if group_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to use the `mlcroissant` library to load, examine, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya. The records were read and referenced by their respective `@id` values as defined in the Croissant schema. Basic exploratory data analysis and visualizations revealed key features and distributions of mental health scores and demographic groupings. Further analysis can be applied to advanced research and public health models.